# E3.2 · Governing autonomy rather than approving tools

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.1 · Translating agentic risk upward](https://spbreed.github.io/cyber-commons/lessons/E3.1.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Write the delegated-authority policy.

**Why a security engineer needs it.** A per-tool review queue becomes a bottleneck and then a bypass. The control it builds is: a policy on delegated authority instead of tool-by-tool approval.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An approved-tools list grows until it is a list of everything, at which point it governs nothing. Autonomy levels and conditions still work at a thousand agents, because they attach to behaviour rather than to product names.

> **At CyberTravels.** CyberTravels will not stop at four agents. An approved-tools list stops governing at about forty; autonomy levels with conditions attach to behaviour and keep working.

## 2 · The framework

```
   autonomy levels, with conditions attached

   L0 proposes only            no conditions
   L1 acts, reversible         logged, sampled
   L2 acts, irreversible       approval + budget + egress control
   L3 acts, no human in loop   L2 + continuous verification + stop authority

   attaches to behaviour, so it still works at a thousand agents
```

Tool-approval processes do not scale for agents. The list of tools grows weekly,
each request needs context the approver does not have, and the queue becomes a
rubber stamp within a quarter.

**Govern autonomy instead.** The autonomy ladder from A1.1 is the right unit
because it is stable — there will always be four rungs — and because it maps
directly onto what can go wrong:

| Rung | Governance |
|---|---|
| **L1** | Self-service. Register it. No further review. |
| **L2** | Register + named owner. Approval gate on every writer, enforced by policy. |
| **L2.5** | Risk tier + blast-radius budget + drift monitoring + tested stop. |
| **L3** | All of L2.5, plus held-out evaluation per release and board sign-off. |

Two properties make this work: a request can be evaluated in minutes by checking
the manifest against the rung, and the policy does not need rewriting when a new
tool appears.

The rung that decides your programme's fate is **L1**. If L1 requires approval,
nobody registers anything and your inventory dies.

## 3 · The procedure, as a skill

Governing the rung rather than the tool. The skill computes each request's blast radius, derives the rung it supports, and refuses with the condition attached — because a bare refusal produces an appeal and a conditional one produces a fix.

In [ ]:
# skills/programme/autonomy-ladder-decisions/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: autonomy-ladder-decisions
description: >-
  Approve or refuse an autonomy request against the rung its blast radius and
  gating actually support, rather than approving the tool it wants. Use when
  agents are being promoted to act unattended.
allowed-tools: Read, Grep, Glob
---

# Govern the rung, not the tool

Approving tools one at a time produces an agent nobody approved: each tool was
reasonable, and the combination acts unattended on production. Governing the
**rung** asks a different question — what does this agent's blast radius and
gating support — and the answer is computed rather than negotiated.

## When to use this

Every request to raise an agent's autonomy, and as a periodic re-check, because
tools accumulate.

## Procedure

**1 — Publish the ladder.** L1 suggests, L2 acts in a sandbox, L2.5 acts with
approval on irreversible actions, L3 acts unattended. With, for each rung, the
governance and the budget it requires.

**2 — Compute the request's blast radius.** Reachable resources weighted by
scope, with gated actions discounted. Ungated writers are what usually decide
the outcome.

**3 — Derive the supported rung from the radius and the gating,** not from the
requester's ask. Then compare. The gap is the conversation.

**4 — Refuse with the condition attached.** "Refused at L2 for ungated writers"
tells the requester what to change. A bare refusal produces an appeal; a
conditional one produces a pull request.

**5 — Re-evaluate on tool change.** A rung approved with three tools does not
carry over to five. Make the tool manifest the trigger for re-evaluation.

## Output contract

```json
{
  "ladder": [{"rung": "str", "governance": "str", "budget": "str"}],
  "requests": [{"name": "str", "asked": "str", "tools": ["str"], "gated": ["str"],
                "blast": 0, "supported": "str", "verdict": "approved|refused", "condition": "str|null"}],
  "reevaluate_on": ["tool added", "scope widened", "model changed"]
}
```

## Failure modes

- **Approving tools.** The combination is what acts.
- **A bare refusal.** Nobody knows what to fix.
- **No re-evaluation trigger.** The manifest grows and the rung does not move.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/programme/autonomy-ladder-decisions/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/programme/autonomy-ladder-decisions/scripts/autonomy_ladder_decisions.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Approve or refuse an autonomy request against the rung its blast radius and gating actually support.

This is the executable half of the `autonomy-ladder-decisions` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

SCOPE_WEIGHT = {"self":1,"project":3,"tenant":8,"org":20}
LADDER = {
 "L1":  "Assist — model proposes, a human performs every action.",
 "L2":  "Act with approval — model calls tools, a human approves each call.",
 "L2.5":"Act within a blast radius — pre-approved tools, bounded scope, review after.",
 "L3":  "Autonomous — model acts and self-verifies; humans see aggregates.",
}
POLICY = {
 "L1":   ("self-service", "register it; no further review", 0),
 "L2":   ("lightweight",  "named owner + approval gate on every writer", 0),
 "L2.5": ("governed",     "risk tier + blast budget + drift monitoring + tested stop", 20),
 "L3":   ("board",        "all of L2.5 + held-out eval per release + board sign-off", 60),
}
for rung, desc in LADDER.items():
    kind, req, budget = POLICY[rung]
    print(f"{rung:5s}{kind:14s}budget {budget:>3}  {req}")
    print(f"{'':19s}{desc}")

def blast(tools, gated):
    return sum(SCOPE_WEIGHT[s]*(1 if rev else 2) for n,s,rev in tools if n not in gated)

def evaluate_request(name, tools, gated, claimed_rung):
    b = blast(tools, gated)
    kind, req, budget = POLICY[claimed_rung]
    problems = []
    writers = [n for n,s,rev in tools if s != "self" or not rev]
    ungated = [n for n in writers if n not in gated]
    if claimed_rung == "L1" and writers:
        problems.append(f"claims L1 but holds state-changing tools: {writers}")
    if claimed_rung == "L2" and ungated:
        problems.append(f"claims L2 (approve every call) but ungated: {ungated}")
    if b > budget:
        problems.append(f"blast radius {b} exceeds the {claimed_rung} budget of {budget}")
    return {"agent": name, "rung": claimed_rung, "blast": b,
            "decision": "APPROVE" if not problems else "REFUSE or RE-TIER",
            "problems": problems, "governance": req}

REQUESTS = [
 ("doc-summariser", [("read_file","self",True)], set(), "L1"),
 ("triage-bot", [("read_file","self",True), ("post_comment","project",True),
                 ("close_ticket","project",True)], set(), "L2"),
 ("refund-agent", [("read_file","self",True),
                   ("issue_refund","tenant",False)], set(), "L2.5"),
 ("refund-agent (gated)", [("read_file","self",True),
                           ("issue_refund","tenant",False)], {"issue_refund"}, "L2.5"),
]
for name, tools, gated, rung in REQUESTS:
    r = evaluate_request(name, tools, gated, rung)
    print(f"{r['agent']:24s}{r['rung']:6s}blast {r['blast']:>3}  {r['decision']}")
    for p in r["problems"]: print(f"{'':30s}⚠ {p}")

import math
def tool_approval_load(n_agents, tools_per_agent, new_tools_per_month,
                       minutes_per_review=25):
    initial = n_agents * tools_per_agent
    monthly = n_agents * new_tools_per_month
    return {"initial_reviews": initial,
            "initial_hours": round(initial * minutes_per_review / 60, 1),
            "monthly_reviews": monthly,
            "monthly_hours": round(monthly * minutes_per_review / 60, 1)}

print(f"{'agents':>8}{'initial reviews':>18}{'hours':>8}{'monthly reviews':>18}{'hours':>8}")
print("-" * 62)
for n in (5, 25, 120):
    r = tool_approval_load(n, tools_per_agent=8, new_tools_per_month=2)
    print(f"{n:>8}{r['initial_reviews']:>18}{r['initial_hours']:>8}"
          f"{r['monthly_reviews']:>18}{r['monthly_hours']:>8}")

def rung_load(n_agents, re_tier_per_month=0.1, minutes=25):
    monthly = n_agents * re_tier_per_month
    return round(monthly * minutes / 60, 1)
print(f"\nsame estates, governing by rung (re-tier only on a manifest change):")
for n in (5, 25, 120):
    print(f"   {n:>4} agents → {rung_load(n)} hours/month")
print("\nTool approval scales with agents × tools. Rung governance scales with")
print("agents × rate of significant change, which is two orders of magnitude less.")

def registration_rate(l1_requires_approval, friction_hours):
    """People register when it is cheaper than not registering."""
    base = 0.95
    penalty = min(friction_hours * 0.35, 0.9)
    return round(base - (penalty if l1_requires_approval else 0.0), 2)

print(f"{'L1 policy':34s}{'friction (h)':>14}{'registration rate':>20}")
print("-" * 70)
for label, approval, hours in (("self-service (register only)", False, 0.1),
                               ("approval required, fast", True, 1.0),
                               ("approval required, committee", True, 40.0)):
    rate = registration_rate(approval, hours)
    print(f"{label:34s}{hours:>14}{rate:>20.0%}")

def inventory_completeness(rate, n_true_assets=120):
    known = int(n_true_assets * rate)
    return {"true_assets": n_true_assets, "registered": known,
            "shadow": n_true_assets - known}

for label, approval, hours in (("self-service", False, 0.1),
                               ("committee", True, 40.0)):
    inv = inventory_completeness(registration_rate(approval, hours))
    print(f"\n{label}: {inv['registered']}/{inv['true_assets']} registered, "
          f"{inv['shadow']} shadow assets")
print("\nEvery control in E1 depends on the inventory. Charging for L1")
print("registration destroys the input to the entire governance programme.")
assert inventory_completeness(registration_rate(True, 40.0))["shadow"] > \
       inventory_completeness(registration_rate(False, 0.1))["shadow"]

## What you just proved

The four rungs print with their governance and budgets. The doc-summariser is approved at L1; `triage-bot` is refused at L2 for ungated writers; the refund agent is refused at L2.5 for exceeding the budget and approved once gated. Tool approval scales to 240 reviews a month at 120 agents against 5 hours for rung governance, and a committee-gated L1 leaves roughly 108 shadow assets.

## Your turn

Write your own per-rung policy in four lines and check what L1 costs a team today. If registering a read-only copilot needs an approval, your inventory is already incomplete and you cannot see by how much.

---

**Next → [E3.3 · Sequencing the programme](https://spbreed.github.io/cyber-commons/lessons/E3.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*